In [ ]:
analysis_year = 2025
traffic_rel_path = "Data/Trafficdata/traffic_25.csv"

In [ ]:
from pathlib import Path
import platform
import re

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / traffic_rel_path).exists() and (project_root.parent / traffic_rel_path).exists():
    project_root = project_root.parent

if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "Malgun Gothic"

plt.rcParams["axes.unicode_minus"] = False


In [ ]:
traffic_path = project_root / traffic_rel_path
festival_path = project_root / "Data/festival.csv"

def extract_festival_dates(festival_df, year):
    year_df = festival_df[festival_df["연도"] == year].copy()
    festival_dates = set()

    for period in year_df["개최기간"].fillna(""):
        text = re.sub(r"\([^)]*\)", "", str(period))
        text = re.sub(r"\s+", "", text)
        parts = re.split(r"[/,]", text)

        for part in parts:
            if not part:
                continue

            if "~" in part:
                match = re.fullmatch(r"(\d{1,2})\.(\d{1,2})\.?~(\d{1,2})\.(\d{1,2})\.?", part)
                if not match:
                    continue
                start_month, start_day, end_month, end_day = map(int, match.groups())
                start_date = pd.Timestamp(year=year, month=start_month, day=start_day)
                end_date = pd.Timestamp(year=year, month=end_month, day=end_day)
                if start_date <= end_date:
                    festival_dates.update(pd.date_range(start_date, end_date, freq="D"))
            else:
                match = re.fullmatch(r"(\d{1,2})\.(\d{1,2})\.?", part)
                if not match:
                    continue
                month, day = map(int, match.groups())
                festival_dates.add(pd.Timestamp(year=year, month=month, day=day))

    return pd.DatetimeIndex(sorted(festival_dates))

traffic_df = pd.read_csv(traffic_path)
traffic_df["일자"] = pd.to_datetime(traffic_df["일자"].astype(str), format="%Y%m%d")
festival_df = pd.read_csv(festival_path)
festival_dates = extract_festival_dates(festival_df, analysis_year)

hour_cols = [col for col in [f"{hour}시" for hour in range(6, 24)] if col in traffic_df.columns]
selected_bridges = ["서강대교", "마포대교", "원효대교"]

filtered_df = traffic_df[
    traffic_df["지점명"].isin(selected_bridges) & (traffic_df["방향"] == "유입")
].copy()

is_weekend = filtered_df["일자"].dt.weekday >= 5
is_festival = filtered_df["일자"].dt.normalize().isin(festival_dates)

filtered_df["날짜타입"] = "평일"
filtered_df.loc[is_weekend, "날짜타입"] = "주말"
filtered_df.loc[is_festival & ~is_weekend, "날짜타입"] = "축제-평일"
filtered_df.loc[is_festival & is_weekend, "날짜타입"] = "축제-주말"

cleaned_df = filtered_df[["지점명", "일자", "날짜타입"] + hour_cols].copy()
display(cleaned_df.head())


In [ ]:
station_avg_df = (
    cleaned_df
    .groupby(["지점명", "날짜타입"], as_index=False)[hour_cols]
    .mean()
)

median_df = station_avg_df.melt(
    id_vars=["지점명", "날짜타입"],
    value_vars=hour_cols,
    var_name="시간",
    value_name="평균유입량",
)
median_df["시간_순서"] = median_df["시간"].str.extract(r"(\d+)").astype(int)
median_df = (
    median_df
    .groupby(["날짜타입", "시간", "시간_순서"], as_index=False)["평균유입량"]
    .median()
    .rename(columns={"평균유입량": "중앙값"})
    .sort_values(["날짜타입", "시간_순서"])
    .reset_index(drop=True)
)

display(station_avg_df.head())
display(median_df.head())


In [ ]:
plot_order = ["평일", "주말", "축제-평일", "축제-주말"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)

for ax, date_type in zip(axes.flatten(), plot_order):
    subset = median_df[median_df["날짜타입"] == date_type].sort_values("시간_순서")

    if subset.empty:
        ax.set_title(f"{date_type} (데이터 없음)")
        ax.axis("off")
        continue

    ax.plot(subset["시간_순서"], subset["중앙값"], marker="o", linewidth=2)
    ax.set_title(date_type)
    ax.set_xlabel("시간대")
    ax.set_ylabel("유입량 중앙값")
    ax.set_xticks(range(6, 24))
    ax.set_xticklabels([f"{hour:02d}시" for hour in range(6, 24)], rotation=45)
    ax.grid(alpha=0.3)

fig.suptitle(f"{analysis_year}년 3개 대교 유입량 중앙값", y=1.02)
fig.tight_layout()
plt.show()
